# Derin Analiz
**EDA bulgularına dayalı istatistiksel doğrulama ve ileri analizler**

---
**İçindekiler**
1. Kütüphane ve veri yükleme
2. İstatistiksel doğrulama (Kruskal-Wallis + Post-hoc)
3. İç Salon yaz düşüşü — sebep araştırması
4. Yağmur günü müşteri kaybı sayısallaştırma
5. Post-COVID davranış değişimi — masa devir hızı
6. Paket segmenti ayrı profil analizi
7. Bahçe doluluk tahmin modeli (Random Forest)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import kruskal, mannwhitneyu
from scikit_posthocs import posthoc_dunn
import os
import warnings
warnings.filterwarnings('ignore')

pio.renderers.default = 'notebook'
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

df_all = pd.read_csv(
    os.path.join('Veriler', 'oturum_hava_birlesik.csv'),
    parse_dates=['acilis_datetime', 'kapama_datetime', 'merge_saati']
)
df = df_all[df_all['outlier_flag'].isna()].copy()

YAGIS_SIRASI = ['Açık', 'Bulutlu', 'Hafif Yağmur', 'Yağmur', 'Sağanak', 'Kar', 'Yoğun Kar']
SICAKLIK_SIRASI = ['Çok Soğuk (<0°)', 'Soğuk (0–10°)', 'Ilık (10–20°)', 'Sıcak (20–30°)', 'Çok Sıcak (>30°)']
df['yagis_kategori'] = pd.Categorical(df['yagis_kategori'], categories=YAGIS_SIRASI, ordered=True)
df['sicaklik_aralik'] = pd.Categorical(df['sicaklik_aralik'], categories=SICAKLIK_SIRASI, ordered=True)

COVID_BASLANGIC = pd.Timestamp('2020-03-11')
COVID_BITIS     = pd.Timestamp('2022-06-01')
df['donem'] = 'Post-COVID'
df.loc[df['acilis_datetime'] < COVID_BASLANGIC, 'donem'] = 'Pre-COVID'
df.loc[(df['acilis_datetime'] >= COVID_BASLANGIC) & (df['acilis_datetime'] <= COVID_BITIS), 'donem'] = 'COVID'
df['tarih_dt'] = pd.to_datetime(df['tarih'])

ay_isimleri = {1:'Oca',2:'Şub',3:'Mar',4:'Nis',5:'May',6:'Haz',
               7:'Tem',8:'Ağu',9:'Eyl',10:'Eki',11:'Kas',12:'Ara'}

print(f'Toplam normal oturum: {len(df):,}')
print(f'Kolonlar: {list(df.columns)}')


## 1. İstatistiksel Doğrulama — Kruskal-Wallis + Dunn Post-hoc
EDA'da gözle fark gördük — şimdi bu farkların istatistiksel olarak anlamlı olup olmadığını test ediyoruz.

In [ ]:
# ── 1A. Yağış kategorisi × oturum süresi Kruskal-Wallis ────────────────────
gruplar_yagis = [
    df[df['yagis_kategori'] == k]['oturum_sure_dk'].dropna().values
    for k in YAGIS_SIRASI
    if len(df[df['yagis_kategori'] == k]) > 30
]
mevcut_kategoriler = [
    k for k in YAGIS_SIRASI
    if len(df[df['yagis_kategori'] == k]) > 30
]

stat_yagis, p_yagis = kruskal(*gruplar_yagis)
print('=== YAĞIŞ × OTURUM SÜRESİ ===')
print(f'Kruskal-Wallis H = {stat_yagis:.2f},  p = {p_yagis:.4e}')
print('→', 'ANLAMLİ FARK VAR (p < 0.05)' if p_yagis < 0.05 else 'Anlamlı fark yok')

print()

# ── 1B. Sıcaklık × oturum süresi Kruskal-Wallis ────────────────────────────
gruplar_sicak = [
    df[df['sicaklik_aralik'] == k]['oturum_sure_dk'].dropna().values
    for k in SICAKLIK_SIRASI
    if len(df[df['sicaklik_aralik'] == k]) > 30
]

stat_sicak, p_sicak = kruskal(*gruplar_sicak)
print('=== SICAKLIK × OTURUM SÜRESİ ===')
print(f'Kruskal-Wallis H = {stat_sicak:.2f},  p = {p_sicak:.4e}')
print('→', 'ANLAMLİ FARK VAR (p < 0.05)' if p_sicak < 0.05 else 'Anlamlı fark yok')

print()

# ── 1C. Dönem × oturum süresi ───────────────────────────────────────────────
gruplar_donem = [
    df[df['donem'] == k]['oturum_sure_dk'].dropna().values
    for k in ['Pre-COVID', 'COVID', 'Post-COVID']
]
stat_donem, p_donem = kruskal(*gruplar_donem)
print('=== DÖNEM × OTURUM SÜRESİ ===')
print(f'Kruskal-Wallis H = {stat_donem:.2f},  p = {p_donem:.4e}')
print('→', 'ANLAMLİ FARK VAR (p < 0.05)' if p_donem < 0.05 else 'Anlamlı fark yok')

# Dönem medyanları
print('\nDönem medyan oturum süreleri:')
for donem in ['Pre-COVID', 'COVID', 'Post-COVID']:
    med = df[df['donem'] == donem]['oturum_sure_dk'].median()
    n   = len(df[df['donem'] == donem])
    print(f'  {donem:12s}: medyan={med:.1f} dk,  n={n:,}')


In [ ]:
# ── 1D. Dunn Post-hoc: Hangi yağış çiftleri farklı? ────────────────────────
df_dunn = df[df['yagis_kategori'].notna()][['yagis_kategori', 'oturum_sure_dk']].dropna()
df_dunn = df_dunn[df_dunn['yagis_kategori'].isin(mevcut_kategoriler)]

dunn_result = posthoc_dunn(
    df_dunn, val_col='oturum_sure_dk', group_col='yagis_kategori',
    p_adjust='bonferroni'
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    dunn_result, annot=True, fmt='.3f', cmap='RdYlGn_r',
    vmin=0, vmax=0.1, ax=ax, linewidths=0.5
)
# Anlamlı olmayan (p >= 0.05) hücreleri noktalı çerçeve ile işaretle
for i in range(len(dunn_result)):
    for j in range(len(dunn_result.columns)):
        if dunn_result.iloc[i, j] >= 0.05:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='gray', lw=2, linestyle='--'))

ax.set_title('Dunn Post-hoc: Yağış Kategorileri Arası p-değerleri (Bonferroni düzeltmeli)\n(Koyu kırmızı = anlamlı fark, p<0.05)')
plt.tight_layout()
plt.show()
print('\nNot: p < 0.05 olan çiftler arasında oturum süresi istatistiksel olarak farklıdır.')


In [ ]:
# ── 1E. Bahçe vs İç Salon Mann-Whitney U (yağışlı vs açık) ─────────────────
print('=== BAHÇE: AÇIK vs YAĞMURLU GÜN OTURUM SÜRESİ ===')
for masa in ['Bahçe', 'İç Salon']:
    acik  = df[(df['masa_grup'] == masa) & (df['yagis_kategori'] == 'Açık')]['oturum_sure_dk'].dropna()
    yagmur = df[(df['masa_grup'] == masa) & (df['yagis_kategori'].isin(['Yağmur', 'Sağanak']))]['oturum_sure_dk'].dropna()
    if len(acik) > 10 and len(yagmur) > 10:
        u, p = mannwhitneyu(acik, yagmur, alternative='two-sided')
        print(f'  {masa}: Açık medyan={acik.median():.1f}dk (n={len(acik):,}) | Yağmur medyan={yagmur.median():.1f}dk (n={len(yagmur):,})')
        print(f'  → Mann-Whitney U={u:.0f}, p={p:.4e}  →  {"ANLAMLİ" if p < 0.05 else "Anlamlı değil"}')
        print()

## 2. İç Salon Yaz Düşüşü — Sebep Araştırması
EDA'da yaz aylarında İç Salon oturumlarının dramatik düştüğünü gördük. Neden?

In [ ]:
# ── 2A. Aylık bazda İç Salon / Bahçe / Toplam oturum trendi ────────────────
aylik_masa = df.groupby(['ay', 'masa_grup']).size().reset_index(name='oturum_sayisi')
aylik_masa['ay_adi'] = aylik_masa['ay'].map(ay_isimleri)
aylik_toplam = df.groupby('ay').size().reset_index(name='toplam')

pivot_ay = aylik_masa.pivot(index='ay', columns='masa_grup', values='oturum_sayisi').fillna(0)
pivot_ay = pivot_ay.merge(aylik_toplam.set_index('ay'), left_index=True, right_index=True)
pivot_ay.index = pivot_ay.index.map(ay_isimleri)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if 'Bahçe' in pivot_ay.columns and 'İç Salon' in pivot_ay.columns:
    ax = axes[0]
    pivot_ay[['Bahçe', 'İç Salon']].plot(kind='bar', ax=ax,
        color=['#2ecc71', '#3498db'], alpha=0.85)
    ax.set_title('Aylık Oturum Sayısı: Bahçe vs İç Salon')
    ax.set_xlabel('')
    ax.set_ylabel('Oturum Sayısı')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='x', rotation=45)
    ax.legend()

    ax2 = axes[1]
    oran = (pivot_ay['İç Salon'] / pivot_ay['toplam'] * 100).round(1)
    oran.plot(kind='bar', ax=ax2, color='#3498db', alpha=0.8)
    ax2.axhline(oran.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Yıllık ort. %{oran.mean():.0f}')
    ax2.set_title('İç Salon\'un Toplam Oturumdaki Payı (%)')
    ax2.set_xlabel('')
    ax2.set_ylabel('%')
    ax2.set_ylim(0, 100)
    ax2.tick_params(axis='x', rotation=45)
    ax2.legend()

    plt.tight_layout()
    plt.show()

    print('\nAylık İç Salon oturum payı (%):')
    print(oran.to_string())
else:
    plt.close()
    print('Uyarı: Veri setinde Bahçe veya İç Salon grubu bulunamadı.')
    print(df['masa_grup'].value_counts().to_string())


In [ ]:
# ── 2B. Yaz aylarında sıcaklık ve İç Salon ilişkisi ────────────────────────
# Günlük ortalama sıcaklık vs İç Salon oturum sayısı scatter
gunluk_ic = df[df['masa_grup'] == 'İç Salon'].groupby('tarih_dt').agg(
    ic_oturum    = ('CEKNO', 'count'),
    ort_sicaklik = ('temperature_2m', 'mean')
).reset_index()

gunluk_ic['ay'] = gunluk_ic['tarih_dt'].dt.month
gunluk_ic['mevsim'] = gunluk_ic['ay'].map({
    12:'Kış', 1:'Kış', 2:'Kış',
    3:'İlkbahar', 4:'İlkbahar', 5:'İlkbahar',
    6:'Yaz', 7:'Yaz', 8:'Yaz',
    9:'Sonbahar', 10:'Sonbahar', 11:'Sonbahar'
})

fig = px.scatter(
    gunluk_ic, x='ort_sicaklik', y='ic_oturum',
    color='mevsim', trendline='lowess',
    title='Günlük Sıcaklık vs İç Salon Oturum Sayısı (Mevsime Göre)',
    labels={'ort_sicaklik': 'Ort. Sıcaklık (°C)', 'ic_oturum': 'İç Salon Oturum Sayısı', 'mevsim': 'Mevsim'},
    color_discrete_map={'Kış':'#3498db', 'İlkbahar':'#2ecc71', 'Yaz':'#e74c3c', 'Sonbahar':'#f39c12'},
    opacity=0.5, height=450
)
fig.show()

# Korelasyon
corr = gunluk_ic['ort_sicaklik'].corr(gunluk_ic['ic_oturum'])
print(f'Sıcaklık × İç Salon oturum korelasyonu: r = {corr:.3f}')
print('→ Negatif korelasyon: Sıcaklık arttıkça İç Salon azalıyor')

In [ ]:
# ── 2C. Yaz aylarında İç Salon + Bahçe toplamı ne kadar?
# Toplam oturumun mevsimsel dağılımı (kapasite mi sorun?)
df['mevsim'] = df['ay'].map({
    12:'Kış', 1:'Kış', 2:'Kış',
    3:'İlkbahar', 4:'İlkbahar', 5:'İlkbahar',
    6:'Yaz', 7:'Yaz', 8:'Yaz',
    9:'Sonbahar', 10:'Sonbahar', 11:'Sonbahar'
})

mevsim_masa = df.groupby(['mevsim', 'masa_grup']).size().reset_index(name='oturum')
mevsim_toplam = df.groupby('mevsim').size().reset_index(name='toplam')
mevsim_masa = mevsim_masa.merge(mevsim_toplam, on='mevsim')
mevsim_masa['pay'] = (mevsim_masa['oturum'] / mevsim_masa['toplam'] * 100).round(1)

MEVSIM_SIRASI = ['Kış', 'İlkbahar', 'Yaz', 'Sonbahar']

fig = px.bar(
    mevsim_masa, x='mevsim', y='oturum',
    color='masa_grup', barmode='stack',
    category_orders={'mevsim': MEVSIM_SIRASI},
    title='Mevsimsel Toplam Oturum Dağılımı (Masa Grubu × Yığılmış)',
    labels={'mevsim': 'Mevsim', 'oturum': 'Oturum Sayısı', 'masa_grup': 'Masa Grubu'},
    text='pay'
)
fig.update_traces(texttemplate='%{text}%', textposition='inside')
fig.show()

print('\nMevsimsel toplam oturum:')
print(mevsim_toplam.set_index('mevsim').reindex(MEVSIM_SIRASI).to_string())

## 3. Yağmur Günü Müşteri Kaybı Sayısallaştırma
Yağışlı günlerde kaç oturum kaybediliyor? Bu kayıp ne kadar gelire denk geliyor?

In [ ]:
# ── 3A. Baz senaryosu: Açık hava günleri ortalama oturum sayısı ─────────────
yagis_gun_ortalama = df.groupby(['tarih_dt', 'yagis_kategori'], observed=True).size().reset_index(name='oturum_sayisi')

baz_acik = yagis_gun_ortalama[yagis_gun_ortalama['yagis_kategori'] == 'Açık']['oturum_sayisi'].mean()

print('=== GÜNLÜK OTURUM BAZLARI ===')
ozet_rows = []
for kat in YAGIS_SIRASI:
    grup = yagis_gun_ortalama[yagis_gun_ortalama['yagis_kategori'] == kat]
    if len(grup) > 0:
        ort = grup['oturum_sayisi'].mean()
        gun_sayisi = len(grup)
        kayip = (baz_acik - ort)
        kayip_pct = (kayip / baz_acik * 100)
        toplam_kayip = kayip * gun_sayisi
        ozet_rows.append({
            'Yağış': kat,
            'Gün Sayısı': gun_sayisi,
            'Ort. Oturum/Gün': round(ort, 1),
            'Açıktan Fark': round(kayip, 1),
            'Kayıp (%)': round(kayip_pct, 1),
            'Toplam Kayıp (tüm yıllar)': round(toplam_kayip)
        })

ozet_df = pd.DataFrame(ozet_rows)
print(ozet_df.to_string(index=False))
print(f'\nBaz (Açık hava günü ort.): {baz_acik:.1f} oturum/gün')

In [ ]:
# ── 3B. Yıllık yağış günü kaybı görselleştirme ──────────────────────────────
fig = px.bar(
    ozet_df[ozet_df['Yağış'] != 'Açık'],
    x='Yağış', y='Toplam Kayıp (tüm yıllar)',
    color='Kayıp (%)',
    color_continuous_scale='RdYlGn_r',
    title='Tüm Yıllar Bazında Yağış Kaynaklı Tahmini Oturum Kaybı',
    labels={'Yağış': 'Yağış Koşulu', 'Toplam Kayıp (tüm yıllar)': 'Tahmini Kayıp (oturum)'},
    text='Toplam Kayıp (tüm yıllar)'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(coloraxis_colorbar_title='Kayıp %')
fig.show()

In [ ]:
# ── 3C. Bahçe kaybı özelinde: Açık günlerde Bahçe beklentisi ne kadar? ─────
yagis_masa_gun = df.groupby(['tarih_dt', 'yagis_kategori', 'masa_grup'], observed=True).size().reset_index(name='oturum_sayisi')

bahce_acik_baz = yagis_masa_gun[
    (yagis_masa_gun['yagis_kategori'] == 'Açık') &
    (yagis_masa_gun['masa_grup'] == 'Bahçe')
]['oturum_sayisi'].mean()

print(f'Açık havada Bahçe ort. günlük oturum: {bahce_acik_baz:.1f}')
print()

print('=== BAHÇE: YAĞIŞ KOŞULUNA GÖRE GÜNLÜK ORT. OTURUM ===')
bahce_rows = []
for kat in YAGIS_SIRASI:
    grup = yagis_masa_gun[
        (yagis_masa_gun['yagis_kategori'] == kat) &
        (yagis_masa_gun['masa_grup'] == 'Bahçe')
    ]
    if len(grup) > 0:
        ort = grup['oturum_sayisi'].mean()
        gun = len(grup)
        kayip_pct = (bahce_acik_baz - ort) / bahce_acik_baz * 100
        bahce_rows.append({'Yağış': kat, 'Gün': gun, 'Ort. Oturum': round(ort, 1), 'Açıktan Kayıp (%)': round(kayip_pct, 1)})
    else:
        bahce_rows.append({'Yağış': kat, 'Gün': 0, 'Ort. Oturum': 0.0, 'Açıktan Kayıp (%)': 100.0})

print(pd.DataFrame(bahce_rows).to_string(index=False))

## 4. Post-COVID Davranış Değişimi — Masa Devir Hızı
Oturum süresi Post-COVID'de düştü. Masa başına günlük kaç müşteri geliyor?

In [ ]:
# ── 4A. Dönem × oturum süresi dağılımı violin plot ─────────────────────────
df_violin = df[df['oturum_sure_dk'] <= 120].copy()

fig, ax = plt.subplots(figsize=(12, 5))
DONEM_SIRASI = ['Pre-COVID', 'COVID', 'Post-COVID']
DONEM_RENK   = {'Pre-COVID': '#3498db', 'COVID': '#e74c3c', 'Post-COVID': '#2ecc71'}

parts = ax.violinplot(
    [df_violin[df_violin['donem'] == d]['oturum_sure_dk'].values for d in DONEM_SIRASI],
    positions=range(len(DONEM_SIRASI)), showmedians=True, showextrema=True
)
for i, (pc, donem) in enumerate(zip(parts['bodies'], DONEM_SIRASI)):
    pc.set_facecolor(DONEM_RENK[donem])
    pc.set_alpha(0.7)

ax.set_xticks(range(len(DONEM_SIRASI)))
ax.set_xticklabels(DONEM_SIRASI)
ax.set_title('Dönemlere Göre Oturum Süresi Dağılımı (0–120 dk)')
ax.set_ylabel('Oturum Süresi (dk)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}'))
plt.tight_layout()
plt.show()

# İstatistikler
print('\nDönem bazlı oturum süresi istatistikleri:')
print(df.groupby('donem')['oturum_sure_dk'].agg(['median', 'mean', 'std', 'count']).round(2).reindex(DONEM_SIRASI).to_string())

In [ ]:
# ── 4B. Dönem × aylık sipariş miktarı ve ürün çeşidi değişimi ─────────────
donem_ozet = df.groupby(['donem', 'ay']).agg(
    ort_sure       = ('oturum_sure_dk', 'median'),
    ort_miktar     = ('toplam_miktar',  'mean'),
    ort_urun_cesit = ('urun_sayisi',    'mean'),
    oturum_sayisi  = ('CEKNO',          'count'),
).reset_index().round(2)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Medyan Oturum Süresi (dk)', 'Ort. Sipariş Miktarı', 'Ort. Ürün Çeşidi'])

renkler = {'Pre-COVID': '#3498db', 'COVID': '#e74c3c', 'Post-COVID': '#2ecc71'}

for donem in DONEM_SIRASI:
    sub = donem_ozet[donem_ozet['donem'] == donem].sort_values('ay')
    sub['ay_adi'] = sub['ay'].map(ay_isimleri)
    for col_idx, col_name in enumerate(['ort_sure', 'ort_miktar', 'ort_urun_cesit'], 1):
        fig.add_trace(
            go.Scatter(x=sub['ay_adi'], y=sub[col_name], mode='lines+markers',
                       name=donem, legendgroup=donem,
                       showlegend=(col_idx == 1),
                       line=dict(color=renkler[donem], width=2),
                       marker=dict(size=6)),
            row=1, col=col_idx
        )

fig.update_layout(title_text='Dönem × Ay Bazında Müşteri Davranışı Karşılaştırması', height=420)
fig.show()

In [ ]:
# ── 4C. Günlük oturum yoğunluğu (masa devir proxy) ─────────────────────────
# Aynı tarihte aynı masaya kaç farklı oturum düştüğünü incele (devir hızı proxy)
if 'masa_no' in df.columns:
    devir = df.groupby(['tarih_dt', 'masa_no', 'donem']).size().reset_index(name='devir_sayisi')
    print('Dönem bazlı ortalama masa devir sayısı (günlük):')
    print(devir.groupby('donem')['devir_sayisi'].agg(['mean', 'median']).round(2).reindex(DONEM_SIRASI).to_string())
else:
    # masa_no yoksa gün bazlı toplam oturum / çalışılan gün sayısı
    gun_oturum = df.groupby(['tarih_dt', 'donem']).size().reset_index(name='oturum_sayisi')
    print('Dönem bazlı ortalama günlük oturum sayısı:')
    print(gun_oturum.groupby('donem')['oturum_sayisi'].agg(['mean', 'median', 'std']).round(2).reindex(DONEM_SIRASI).to_string())

# Pre vs Post dönem karşılaştırmalı kutu grafiği
gun_oturum = df.groupby(['tarih_dt', 'donem']).size().reset_index(name='oturum_sayisi')
gun_oturum = gun_oturum[gun_oturum['donem'].isin(['Pre-COVID', 'Post-COVID'])]

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=gun_oturum, x='donem', y='oturum_sayisi',
            order=['Pre-COVID', 'Post-COVID'],
            palette={'Pre-COVID': '#3498db', 'Post-COVID': '#2ecc71'}, ax=ax)
ax.set_title('Pre-COVID vs Post-COVID: Günlük Oturum Sayısı Dağılımı')
ax.set_xlabel('')
ax.set_ylabel('Günlük Oturum Sayısı')

# Mann-Whitney test
pre  = gun_oturum[gun_oturum['donem'] == 'Pre-COVID']['oturum_sayisi']
post = gun_oturum[gun_oturum['donem'] == 'Post-COVID']['oturum_sayisi']
u, p = mannwhitneyu(pre, post, alternative='two-sided')
ax.set_xlabel(f'p = {p:.4e} ({"Anlamlı fark" if p < 0.05 else "Anlamlı değil"})', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Paket Segmenti Ayrı Profil Analizi
Scatter grafiğinde Paket segmenti çok farklı davranış gösterdi — kim bu müşteriler?

In [ ]:
# ── 5A. Paket vs diğer gruplar temel istatistikler ─────────────────────────
if 'masa_grup' in df.columns:
    paket_ozet = df.groupby('masa_grup').agg(
        oturum_sayisi  = ('CEKNO',          'count'),
        ort_sure_dk    = ('oturum_sure_dk',  'mean'),
        medyan_sure_dk = ('oturum_sure_dk',  'median'),
        ort_miktar     = ('toplam_miktar',   'mean'),
        ort_urun       = ('urun_sayisi',     'mean'),
    ).round(2)
    print('Masa Grubu Karşılaştırması:')
    print(paket_ozet.to_string())

# Dağılım karşılaştırma
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, (col, title) in enumerate([('oturum_sure_dk', 'Oturum Süresi (dk)'),
                                    ('toplam_miktar', 'Sipariş Miktarı'),
                                    ('urun_sayisi', 'Ürün Çeşidi')]):
    df_plot = df[df[col] > 0].copy()
    for masa in df['masa_grup'].dropna().unique():
        vals = df_plot[df_plot['masa_grup'] == masa][col].dropna()
        if len(vals) > 100:
            vals.plot.kde(ax=axes[i], label=masa, linewidth=2)
    if col == 'oturum_sure_dk':
        axes[i].set_xlim(0, 120)
    axes[i].set_title(title)
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=9)

plt.suptitle('Masa Grubu Bazında Dağılım Karşılaştırması (KDE)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5B. Paket segmentinin zamansal profili ──────────────────────────────────
paket_df = df[df['masa_grup'] == 'Paket'].copy() if 'Paket' in df['masa_grup'].values else pd.DataFrame()

if not paket_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Saatlik dağılım
    paket_saat = paket_df.groupby('saat').size().reset_index(name='sayi')
    sns.barplot(data=paket_saat, x='saat', y='sayi', palette='rocket', ax=axes[0])
    axes[0].set_title('Paket: Saatlik Dağılım')
    axes[0].set_xlabel('Saat')

    # Aylık dağılım
    paket_ay = paket_df.groupby('ay').size().reset_index(name='sayi')
    paket_ay['ay_adi'] = paket_ay['ay'].map(ay_isimleri)
    sns.barplot(data=paket_ay, x='ay_adi', y='sayi', palette='rocket', ax=axes[1])
    axes[1].set_title('Paket: Aylık Dağılım')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)

    # Gün bazlı dağılım
    GUN_SIRASI = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    GUN_TR = {'Monday':'Pzt','Tuesday':'Sal','Wednesday':'Çar','Thursday':'Per',
               'Friday':'Cum','Saturday':'Cmt','Sunday':'Paz'}
    paket_gun = paket_df.groupby('gun_adi').size().reset_index(name='sayi')
    paket_gun['gun_adi'] = pd.Categorical(paket_gun['gun_adi'], categories=GUN_SIRASI, ordered=True)
    paket_gun = paket_gun.sort_values('gun_adi')
    paket_gun['gun_tr'] = paket_gun['gun_adi'].map(GUN_TR)
    sns.barplot(data=paket_gun, x='gun_tr', y='sayi', palette='rocket', ax=axes[2])
    axes[2].set_title('Paket: Günlük Dağılım')
    axes[2].set_xlabel('')

    plt.suptitle('Paket Segmenti Zaman Profili', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Paket grubu bulunamadı, masa_grup sütunundaki değerleri kontrol et:')
    print(df['masa_grup'].value_counts().to_string())

In [ ]:
# ── 5C. Paket: Hava koşulu hassasiyeti var mı? ──────────────────────────────
if not paket_df.empty:
    paket_yagis = paket_df.groupby('yagis_kategori', observed=True).agg(
        oturum_sayisi = ('CEKNO', 'count'),
        ort_sure      = ('oturum_sure_dk', 'mean'),
    ).reset_index().round(2)

    ic_yagis = df[df['masa_grup'] == 'İç Salon'].groupby('yagis_kategori', observed=True).agg(
        oturum_sayisi = ('CEKNO', 'count'),
        ort_sure      = ('oturum_sure_dk', 'mean'),
    ).reset_index().round(2)

    fig = make_subplots(rows=1, cols=2, subplot_titles=['Paket: Yağış × Oturum Sayısı', 'İç Salon: Yağış × Oturum Sayısı (Karşılaştırma)'])
    for col_idx, (data, isim) in enumerate([(paket_yagis, 'Paket'), (ic_yagis, 'İç Salon')], 1):
        data = data.copy()
        max_val = data['oturum_sayisi'].max()
        data['normalize'] = (data['oturum_sayisi'] / max_val * 100).round(1)
        fig.add_trace(
            go.Bar(x=data['yagis_kategori'].astype(str), y=data['normalize'],
                   text=data['normalize'], texttemplate='%{text}%',
                   name=isim, showlegend=True),
            row=1, col=col_idx
        )
    fig.update_layout(title_text='Normalize Edilmiş Yağış Hassasiyeti (Maks=100%)', height=400)
    fig.show()


## 6. Bahçe Doluluk Tahmin Modeli (Random Forest)
Hava durumu + tarih bilgisiyle bir sonraki günün Bahçe oturum sayısını tahmin et.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── 6A. Model için günlük Bahçe veri seti hazırlama ─────────────────────────
bahce_gunluk = df[df['masa_grup'] == 'Bahçe'].groupby('tarih_dt').agg(
    bahce_oturum = ('CEKNO', 'count'),
).reset_index()

# Hava bilgilerini günlük bazda al (tüm oturumlardan)
hava_gunluk = df.groupby('tarih_dt').agg(
    ort_sicaklik  = ('temperature_2m', 'mean'),
    yagis_kodu    = ('yagis_kategori', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    toplam_oturum = ('CEKNO', 'count'),
).reset_index()

hava_gunluk['yil']        = hava_gunluk['tarih_dt'].dt.year
hava_gunluk['ay']         = hava_gunluk['tarih_dt'].dt.month
hava_gunluk['gun']        = hava_gunluk['tarih_dt'].dt.dayofweek  # 0=Pzt, 6=Paz
hava_gunluk['hafta_sonu'] = (hava_gunluk['gun'] >= 5).astype(int)
hava_gunluk['mevsim']     = hava_gunluk['ay'].map({
    12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3
})

# Bahçe oturum sayısı ile birleştir (0 ile doldur — bahçe kapalı günler)
model_df = hava_gunluk.merge(bahce_gunluk[['tarih_dt', 'bahce_oturum']], on='tarih_dt', how='left')
model_df['bahce_oturum'] = model_df['bahce_oturum'].fillna(0)

# Yağış kategorisini encode et
le = LabelEncoder()
model_df['yagis_encode'] = le.fit_transform(model_df['yagis_kodu'].astype(str))

print(f'Model veri seti: {len(model_df):,} gün')
print(f'Ortalama günlük Bahçe oturumu: {model_df["bahce_oturum"].mean():.1f}')
print(f'Sıfır Bahçe oturumu olan günler: {(model_df["bahce_oturum"]==0).sum():,}')
display(model_df.head())


In [ ]:
# ── 6B. Random Forest Eğitim ────────────────────────────────────────────────
FEATURES = ['ort_sicaklik', 'yagis_encode', 'ay', 'gun', 'hafta_sonu', 'mevsim', 'yil']
TARGET   = 'bahce_oturum'

model_clean = model_df[FEATURES + [TARGET]].dropna()
X = model_clean[FEATURES]
y = model_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
# CV skoru
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

print('=== MODEL PERFORMANSI ===')
print(f'Test MAE  : {mae:.2f} oturum/gün')
print(f'Test R²   : {r2:.3f}')
print(f'CV MAE    : {-cv_scores.mean():.2f} ± {cv_scores.std():.2f}')
print(f'\nYorum: Model günlük Bahçe oturumunu ortalama ±{mae:.0f} oturum hatayla tahmin ediyor.')

In [ ]:
# ── 6C. Feature Importance ──────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'Özellik': FEATURES,
    'Önem': rf.feature_importances_
}).sort_values('Önem', ascending=True)

# Türkçe etiketler
etiket_map = {
    'ort_sicaklik': 'Ortalama Sıcaklık',
    'yagis_encode': 'Yağış Koşulu',
    'ay': 'Ay',
    'gun': 'Haftanın Günü',
    'hafta_sonu': 'Hafta Sonu',
    'mevsim': 'Mevsim',
    'yil': 'Yıl'
}
importance_df['Özellik'] = importance_df['Özellik'].map(etiket_map)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(importance_df['Özellik'], importance_df['Önem'],
               color=['#e74c3c' if x > 0.2 else '#3498db' for x in importance_df['Önem']])
ax.set_xlabel('Feature Importance')
ax.set_title('Bahçe Doluluk Tahmininde Değişkenlerin Önemi (Random Forest)')
for bar, val in zip(bars, importance_df['Önem']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── 6D. Gerçek vs Tahmin görselleştirme ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: gerçek vs tahmin
axes[0].scatter(y_test, y_pred, alpha=0.3, color='#3498db', s=15)
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Mükemmel tahmin')
axes[0].set_xlabel('Gerçek')
axes[0].set_ylabel('Tahmin')
axes[0].set_title(f'Gerçek vs Tahmin (R²={r2:.3f})')
axes[0].legend()

# Hata dağılımı
hatalar = y_pred - y_test.values
axes[1].hist(hatalar, bins=50, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Hata (Tahmin − Gerçek)')
axes[1].set_ylabel('Frekans')
axes[1].set_title(f'Tahmin Hatası Dağılımı (MAE={mae:.1f})')

plt.tight_layout()
plt.show()

# Örnek tahminler
print('\n=== ÖRNEK TAHMİNLER ===')
ornek = pd.DataFrame({
    'Senaryo'  : ['Sıcak Pazar (Haz)', 'Yağmurlu Çarşamba (Kas)', 'Karlı Ocak Sabahı', 'Açık Cuma Öğlesi'],
    'ort_sicaklik': [28, 12, -2, 22],
    'yagis_encode': [
        le.transform(['Açık'])[0],
        le.transform(['Yağmur'])[0],
        le.transform(['Kar'])[0] if 'Kar' in le.classes_ else 0,
        le.transform(['Açık'])[0]
    ],
    'ay'         : [6, 11, 1, 5],
    'gun'        : [6, 2, 0, 4],
    'hafta_sonu' : [1, 0, 0, 0],
    'mevsim'     : [2, 3, 0, 1],
    'yil'        : [2024, 2024, 2024, 2024]
})
ornek['Tahmin Edilen Bahçe Oturumu'] = rf.predict(ornek[FEATURES]).round(0).astype(int)
print(ornek[['Senaryo', 'Tahmin Edilen Bahçe Oturumu']].to_string(index=False))